In [1]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
torch.manual_seed(42)


In [2]:
ATTACK_FILE = "C:\\Users\\johfr\\Dropbox\\Skule\\Data science\\Bachelor\\Dataset\\Brute Force\\combined_brute_force.csv"   # change per attack
BENIGN_FILE = "C:\\Users\\johfr\\Dropbox\\Skule\\Data science\\Bachelor\\Dataset\\Benign\\combined_benign.csv"
MODEL_PATH   = "brute_force_SVM.pth" 
SAMPLE_CAP  = 50000
RANDOM_SEED = 42

df_attack = pd.read_csv(ATTACK_FILE)
df_attack = df_attack.sample(n=min(len(df_attack), SAMPLE_CAP), random_state=RANDOM_SEED)

df_benign = pd.read_csv(BENIGN_FILE)
df_benign = df_benign.sample(n=len(df_attack), random_state=RANDOM_SEED)

# Combine and shuffle
df = pd.concat([df_benign, df_attack], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED)

X = df.drop(columns=["Label"])
y = df["Label"]

# 60/20/20 split with stratification
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=RANDOM_SEED, stratify=y)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp)

print(f"Attack: {ATTACK_FILE} | Total rows: {len(df)}")
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Attack: C:\Users\johfr\Dropbox\Skule\Data science\Bachelor\Dataset\Brute Force\combined_brute_force.csv | Total rows: 7238
Train: 4342 | Val: 1448 | Test: 1448


In [3]:
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_val   = X_val.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test  = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

In [4]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

C = 1.0  # regularization strength — lower = stronger regularization
svm = LinearSVC(C=C, max_iter=2000, random_state=42)

In [5]:
svm.fit(X_train_scaled, y_train)
print(f"SVM trained with C={C}")

# Validate
val_preds = svm.predict(X_val_scaled)
print(classification_report(y_val, val_preds, target_names=["Benign", "Attack"]))

SVM trained with C=1.0
              precision    recall  f1-score   support

      Benign       0.82      0.56      0.66       724
      Attack       0.66      0.88      0.76       724

    accuracy                           0.72      1448
   macro avg       0.74      0.72      0.71      1448
weighted avg       0.74      0.72      0.71      1448



In [6]:
preds = svm.predict(X_test_scaled)

print(classification_report(y_test, preds, target_names=["Benign", "Attack"]))
print(confusion_matrix(y_test, preds))

              precision    recall  f1-score   support

      Benign       0.83      0.58      0.68       724
      Attack       0.68      0.88      0.76       724

    accuracy                           0.73      1448
   macro avg       0.75      0.73      0.72      1448
weighted avg       0.75      0.73      0.72      1448

[[417 307]
 [ 86 638]]
